# 04 — Textualize: Build LLM Prompt Dataset (V10 enriched — YoY + corridor ranking)

**Branch:** `v10_enriched_yoy_corridors`

This notebook is the **forecast-first textualization branch** for V9. We are deliberately moving away from broad conversational supervision and pushing the model toward a cleaner forecasting task.

**V10 objectives:**
1. Keep V9 traffic-memory base intact
2. Add **year-over-year (YoY) comparison** for 2024 rows vs same month 2023
3. Add **inter-corridor ranking** (where does this corridor sit vs the other 3 this month?)
4. Richer context → better grounding and explanation quality

**Training design in V9:**
- train split: **forecast only**
- forecast variants: `full`, `forecast_traffic_core`, `forecast_lag_focus`, `forecast_profile_only`, `forecast_minimal_json`
- val/test: keep all question types for evaluation coverage

**Canonical forecast target:**
```json
{"next_month_am_tt_ratio": 1.250, "delta_vs_current": -0.010}
```

The core idea is simple: if we want better forecast behavior, we give the model less narrative clutter and more structured traffic state + traffic memory.


In [1]:
import pandas as pd
import numpy as np
import json
import os
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 120)

print('Libraries loaded.')
print(f'Pandas {pd.__version__} | NumPy {np.__version__}')

Libraries loaded.
Pandas 3.0.2 | NumPy 2.4.4


In [2]:
# ── Load ml_dataset ────────────────────────────────────────────────────────────
df = pd.read_csv('../data/ml_dataset_gpu.csv')

print('=' * 70)
print('ml_dataset_gpu.csv LOADED')
print('=' * 70)
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Missing values (NaN lags Jan-Mar 2023): {df.isna().sum().sum()}')
print(f'Splits: {dict(df["split"].value_counts())}')

lag_cols = [c for c in df.columns if c.endswith(('_l1', '_l2', '_l3'))]
print(f'Lag columns ({len(lag_cols)}): {lag_cols}')

print()
print('Rows per corridor:')
for cid in sorted(df['corr_id'].unique()):
    n = (df['corr_id'] == cid).sum()
    name = df[df['corr_id'] == cid]['corridor'].iloc[0]
    print(f'  [{cid}] {name}: {n} rows')

print()
print('Temporal range:', f"{df['year'].min()}-{df['month'].min():02d}", '-->', f"{df['year'].max()}-{df['month'].max():02d}")

ml_dataset_gpu.csv LOADED
Shape: 96 rows x 132 cols
Missing values (NaN lags Jan-Mar 2023): 240
Splits: {'train': np.int64(48), 'val': np.int64(24), 'test': np.int64(24)}
Lag columns (30): ['flt_total_pax_l1', 'flt_total_pax_l2', 'flt_total_pax_l3', 'wx_rain_mm_sum_l1', 'wx_rain_mm_sum_l2', 'wx_rain_mm_sum_l3', 'soc_phuket_l1', 'soc_phuket_l2', 'soc_phuket_l3', 'cal_n_holidays_l1', 'cal_n_holidays_l2', 'cal_n_holidays_l3', 'tt_ratio_Weekday_AM1_l1', 'tt_ratio_Weekday_AM1_l2', 'tt_ratio_Weekday_AM1_l3', 'pti_Weekday_AM1_l1', 'pti_Weekday_AM1_l2', 'pti_Weekday_AM1_l3', 'spd_Weekday_AM1_l1', 'spd_Weekday_AM1_l2', 'spd_Weekday_AM1_l3', 'tt_ratio_Weekday_AM2_l1', 'tt_ratio_Weekday_AM2_l2', 'tt_ratio_Weekday_AM2_l3', 'tt_ratio_Weekday_PrePM_l1', 'tt_ratio_Weekday_PrePM_l2', 'tt_ratio_Weekday_PrePM_l3', 'tt_ratio_Weekend_AM1_l1', 'tt_ratio_Weekend_AM1_l2', 'tt_ratio_Weekend_AM1_l3']

Rows per corridor:
  [0] Airport Road (Route 402): 24 rows
  [1] Patong Hill (Route 4029): 24 rows
  [2] Phuke

In [3]:
# ── Corridor metadata and helper functions ─────────────────────────────────────
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
MONTH_NAMES_FULL = ['January','February','March','April','May','June',
                    'July','August','September','October','November','December']

CORR_META = {
    0: {'name':'Airport Road (Route 402)',
        'short':'Airport Road',
        'desc':'Thepkrasattri Road from HKT Airport to Thalang — primary tourist arrival corridor, directly sensitive to flight schedules.',
        'risk':'tourist arrival surges; airport-linked demand spikes',
        'alt':'Route 4026 (parallel local road)'},
    1: {'name':'Patong Hill (Route 4029)',
        'short':'Patong Hill',
        'desc':'Mountain pass road from Kathu to Patong Beach — most congested corridor on the island, strong high/low season contrast.',
        'risk':'mountain pass susceptible to flooding; PM peak bottleneck; no bypass alternative',
        'alt':'Bypass Road (Route 4027) then Kamala coastal road'},
    2: {'name':'Phuket Town to Rawai (Route 4022)',
        'short':'Town-Rawai',
        'desc':'Urban corridor from Phuket Town center to southern beaches (Nai Harn, Rawai) — mix of residential and beach tourist traffic.',
        'risk':'town center intersection congestion; highest PTI on island',
        'alt':'Chao Fa West Road (Route 4028)'},
    3: {'name':'Bypass Road (Route 4027)',
        'short':'Bypass Road',
        'desc':'Eastern ring road from Kathu to Chalong — strategic bypass/reference corridor, but still congested in peak months.',
        'risk':'ring-road peak congestion and event spillovers; useful comparator, not always uncongested',
        'alt':'Primary route (already a bypass)'},
}

SEASON_FULL = {
    'high_peak': 'high-season peak (Dec-Mar: dry weather, maximum tourist influx)',
    'shoulder':  'shoulder season (Apr-May or Nov: transitioning months)',
    'monsoon':   'monsoon season (Jun-Oct: heavy rain, reduced tourism)',
    # legacy keys
    'high':      'high season (Nov or Mar-Apr: strong tourist demand)',
    'low_pre':   'shoulder pre-monsoon (Apr-May: transitioning to rainy season)',
    'low_post':  'shoulder post-monsoon (Oct-Nov: transitioning to dry season)',
    'low':       'low / monsoon season (Jun-Sep: heavy rain, reduced tourism)',
}

def tt_level(tt_ratio):
    'Traffic level from TomTom travel time ratio (1.0=free-flow; higher=slower/more congested).'
    if tt_ratio <= 1.05: return 'very light'
    elif tt_ratio <= 1.20: return 'light'
    elif tt_ratio <= 1.40: return 'moderate'
    elif tt_ratio <= 1.60: return 'heavy'
    else: return 'very heavy'

def cong_label(c):
    if c < 0.25: return 'low'
    elif c < 0.45: return 'moderate'
    elif c < 0.65: return 'high'
    else: return 'very high'

def pti_desc(pti_max):
    if pti_max < 1.5: return 'reliable journey times'
    elif pti_max < 2.0: return 'moderate variability — allow 10-15 min buffer'
    elif pti_max < 2.5: return 'high variability — allow 20-35 min buffer'
    else: return 'very high variability — worst journeys 2.5-3x free-flow duration'

def rain_note(rain_mm, is_monsoon):
    if rain_mm > 300: return f'{rain_mm:.0f} mm (heavy monsoon rainfall — wet roads, reduced visibility)'
    elif rain_mm > 150: return f'{rain_mm:.0f} mm (significant rain — plan for congestion at flood-prone spots)'
    elif rain_mm > 50: return f'{rain_mm:.0f} mm (moderate showers — minor road impact)'
    else: return f'{rain_mm:.0f} mm (dry / light rain — negligible road impact)'

def get_next_month_row(df_ref, corr_id, year, month):
    'Get the next calendar month row for this corridor, or None if not in dataset.'
    nm, ny = (month % 12) + 1, year + (1 if month == 12 else 0)
    r = df_ref[(df_ref['corr_id'] == corr_id) & (df_ref['year'] == ny) & (df_ref['month'] == nm)]
    return r.iloc[0] if len(r) > 0 else None

def get_nth_month_row(df_ref, corr_id, year, month, n):
    'Get the row n months ahead for this corridor, or None.'
    m, y = month, year
    for _ in range(n):
        m += 1
        if m > 12: m = 1; y += 1
    r = df_ref[(df_ref['corr_id'] == corr_id) & (df_ref['year'] == y) & (df_ref['month'] == m)]
    return r.iloc[0] if len(r) > 0 else None

def next_month_label(year, month, n):
    m, y = month, year
    for _ in range(n): m += 1;  y += 1 if m > 12 else 0;  m = m if m <= 12 else 1
    return f'{MONTH_NAMES_FULL[m-1]} {y}'

print('Corridor metadata and helper functions defined.')
print(f'CORR_META keys: {list(CORR_META.keys())}')
print(f'SEASON_FULL keys: {list(SEASON_FULL.keys())}')

# Quick validation
test_vals = [1.00, 1.10, 1.30, 1.50, 1.80, 2.05]
print('\nTraffic level mapping validation:')
for v in test_vals:
    print(f'  tt_ratio={v:.3f} -> {tt_level(v)}')

def get_yoy_row(df_ref, corr_id, year, month):
    'Get the same corridor+month from the previous year, or None if not available.'
    r = df_ref[(df_ref['corr_id'] == corr_id) & (df_ref['year'] == year - 1) & (df_ref['month'] == month)]
    return r.iloc[0] if len(r) > 0 else None


def get_corridor_ranking(df_ref, year, month):
    'Return all 4 corridors for this month sorted by AM1 tt_ratio ascending (best first).'
    rows = df_ref[(df_ref['year'] == year) & (df_ref['month'] == month)]
    result = []
    for _, row in rows.iterrows():
        result.append({
            'corr_id': int(row['corr_id']),
            'name': CORR_META[int(row['corr_id'])]['short'],
            'tt_am1': float(row['tt_ratio_Weekday_AM1']),
            'pti_max': float(row['pti_max']),
        })
    return sorted(result, key=lambda x: x['tt_am1'])


Corridor metadata and helper functions defined.
CORR_META keys: [0, 1, 2, 3]
SEASON_FULL keys: ['high_peak', 'shoulder', 'monsoon', 'high', 'low_pre', 'low_post', 'low']

Traffic level mapping validation:
  tt_ratio=1.000 -> very light
  tt_ratio=1.100 -> light
  tt_ratio=1.300 -> moderate
  tt_ratio=1.500 -> heavy
  tt_ratio=1.800 -> very heavy
  tt_ratio=2.050 -> very heavy


In [4]:
# -- 6 Answer generators -------------------------------------------------------

def _signed_delta_text(delta):
    if delta > 0.03:
        return f'increase of {delta:.3f}'
    if delta < -0.03:
        return f'decrease of {abs(delta):.3f}'
    return f'near-stable change of {delta:+.3f}'


def _seasonal_outlook_label(month_num):
    if month_num in [11, 12, 1, 2, 3]:
        return 'high-season pattern'
    if month_num in [5, 6, 7, 8, 9, 10]:
        return 'monsoon pattern'
    return 'shoulder-season pattern'


def answer_nowcast(row, df_ref):
    r = row
    cid = int(r['corr_id'])
    meta = CORR_META[cid]
    mn = MONTH_NAMES_FULL[int(r['month']) - 1]
    yr = int(r['year'])
    season = SEASON_FULL.get(r['season'], r['season'])

    tt_am  = r['tt_ratio_Weekday_AM1'];  tt_pm  = r['tt_ratio_Weekday_PM1']
    tt_off = r['tt_ratio_Weekday_Midday'];  tt_wk  = r['tt_ratio_Weekend_AM1']
    sp_am  = r['spd_Weekday_AM1']; sp_pm  = r['spd_Weekday_PM1']
    sp_off = r['spd_Weekday_Midday']; sp_wk  = r['spd_Weekend_AM1']
    pti_m  = r['pti_mean'];             pti_mx = r['pti_max']
    pax    = int(r['flt_total_pax']);   rain   = r['wx_rain_mm_sum']
    temp   = r['wx_temp_c_mean'];       wind   = r['wx_wind_kmh']
    soc    = r['soc_phuket'];           hol    = int(r['cal_n_holidays'])
    evt    = int(r['cal_n_events'])

    worst_level = tt_level(max(tt_am, tt_pm, tt_off, tt_wk))

    txt = (
        f'Traffic conditions on {meta["name"]} - {mn} {yr} ({season}):\n\n'
        f'Travel Time Ratios (TomTom - 1.0=free-flow, higher=slower/more congested):\n'
        f'  AM Peak  (Mon-Fri 07h): {tt_am:.3f} [{tt_level(tt_am)}] | Speed: {sp_am:.1f} km/h\n'
        f'  PM Peak  (Mon-Fri 17h): {tt_pm:.3f} [{tt_level(tt_pm)}] | Speed: {sp_pm:.1f} km/h\n'
        f'  Midday   (Mon-Fri 12-13h): {tt_off:.3f} [{tt_level(tt_off)}] | Speed: {sp_off:.1f} km/h\n'
        f'  Weekend  (Sat-Sun 07h): {tt_wk:.3f} [{tt_level(tt_wk)}] | Speed: {sp_wk:.1f} km/h\n\n'
        f'Planning Time Index: {pti_m:.2f} (mean), {pti_mx:.2f} (max) - {pti_desc(pti_mx)}\n\n'
        f'Contextual drivers - {mn} {yr}:\n'
        f'  Tourism : {pax:,} total passengers at HKT airport\n'
        f'  Weather : {rain_note(rain, int(r["is_monsoon"]))} | {temp:.1f}C avg | {wind:.1f} km/h wind\n'
        f'  Trends  : Google Trends "Phuket" = {soc:.0f}/100\n'
        f'  Calendar: {hol} public holiday(s), {evt} event(s)\n\n'
        f'Overall: Traffic on {meta["short"]} is {worst_level} in {mn} {yr}. '
        f'The {"monsoon season suppresses tourism and traffic volumes" if int(r["is_monsoon"]) else "seasonal pattern drives congestion peaks at AM and PM rush hours"}.\n'
    )
    return txt


def answer_forecast(row, df_ref, variant='full'):
    r = row
    cid = int(r['corr_id'])
    yr = int(r['year'])
    month = int(r['month'])

    current_tt = float(r['tt_ratio_Weekday_AM1'])
    next_row = get_nth_month_row(df_ref, cid, yr, month, 1)
    if next_row is not None:
        next_tt = float(next_row['tt_ratio_Weekday_AM1'])
    else:
        next_tt = current_tt
    delta = next_tt - current_tt

    # V9 canonical target: short, machine-readable, forecast-only.
    return f'{{"next_month_am_tt_ratio": {next_tt:.3f}, "delta_vs_current": {delta:+.3f}}}'

def answer_explain(row, df_ref, variant='full'):
    r = row
    cid = int(r['corr_id'])
    meta = CORR_META[cid]
    mn = MONTH_NAMES_FULL[int(r['month']) - 1]
    yr = int(r['year'])

    pax = int(r['flt_total_pax'])
    rain = float(r['wx_rain_mm_sum'])
    temp = float(r['wx_temp_c_mean'])
    wind = float(r['wx_wind_kmh'])
    soc = float(r['soc_phuket'])
    hol = int(r['cal_n_holidays'])
    evt = int(r['cal_n_events'])
    high_imp = bool(int(r['cal_high_impact']))
    is_monsoon = int(r['is_monsoon'])
    season = r['season']

    tt_am = float(r['tt_ratio_Weekday_AM1'])
    tt_pm = float(r['tt_ratio_Weekday_PM1'])
    tt_mid = float(r['tt_ratio_Weekday_Midday'])
    tt_wk = float(r['tt_ratio_Weekend_AM1'])
    pti_mx = float(r['pti_max'])

    worst_tt = max(tt_am, tt_pm)
    level = tt_level(worst_tt)

    sections = []
    traffic_score = abs(tt_pm - tt_am) + abs(worst_tt - 1.0) + pti_mx / 3
    sections.append((
        traffic_score,
        'OBSERVED TRAFFIC PROFILE',
        f'AM={tt_am:.3f}, PM={tt_pm:.3f}, Midday={tt_mid:.3f}, Weekend={tt_wk:.3f}. PM is the main bottleneck and PTI max={pti_mx:.2f} indicates {pti_desc(pti_mx)}.'
    ))

    corridor_score = 1.4 + (0.4 if int(r['corr_tourist_access']) else 0.0)
    sections.append((
        corridor_score,
        'CORRIDOR STRUCTURE',
        f'{meta["desc"]} Key risk: {meta["risk"]}.'
    ))

    if variant in ('full', 'flow_cal_wx'):
        weather_score = abs(rain - 120) / 140 + wind / 35
        weather_note = 'Weather is likely amplifying delay and reliability pressure.' if rain > 120 or wind > 15 else 'Weather is present but is probably a secondary driver this month.'
        sections.append((
            weather_score,
            'WEATHER CONDITIONS',
            f'{rain_note(rain, is_monsoon)} | {temp:.1f}C avg | {wind:.1f} km/h wind. {weather_note}'
        ))

    if variant in ('full', 'flow_cal_wx', 'flow_cal'):
        calendar_score = (2.0 if high_imp else 0.0) + hol * 0.25 + evt * 0.12
        cal_note = 'Calendar pressure is meaningful and can create local spikes.' if (high_imp or hol >= 3 or evt >= 6) else 'Calendar pressure looks limited, so the baseline pattern is mostly structural.'
        sections.append((
            calendar_score,
            'CALENDAR PRESSURE',
            f'{hol} holiday(s), {evt} event(s), high-impact={"yes" if high_imp else "no"}. {cal_note}'
        ))

    if variant == 'full':
        tourism_score = abs(pax - 800000) / 300000 + abs(soc - 50) / 25
        season_note = 'Peak-season demand is likely pushing corridor usage higher.' if season in ['high_peak', 'high'] else ('Monsoon demand is softer, but traffic can remain uneven on tourist corridors.' if is_monsoon else 'Shoulder-season demand keeps traffic moderate unless another driver dominates.')
        sections.append((
            tourism_score,
            'SEASON AND DEMAND',
            f'{SEASON_FULL.get(season, season)} | HKT passengers={pax:,} | Google Trends={soc:.0f}/100. {season_note}'
        ))

    if variant == 'flow_only':
        sections.append((
            pti_mx / 3,
            'RELIABILITY SIGNAL',
            f'PTI max={pti_mx:.2f} shows how far worst-case trips move away from free-flow even when the average level looks manageable.'
        ))

    sections = sorted(sections, key=lambda x: x[0], reverse=True)
    top_sections = sections[:3]

    txt = (
        f'Explanation of traffic on {meta["name"]} in {mn} {yr}:\n\n'
        f'Current traffic status: {level} (TT ratio AM={tt_am:.3f}, PM={tt_pm:.3f}; PTI max={pti_mx:.2f})\n\n'
        f'Key contributing factors (visible in the prompt):\n\n'
    )

    for rank, (_, title, body) in enumerate(top_sections, 1):
        txt += f'{rank}. {title}\n   {body}\n\n'

    txt += f'Summary: current congestion on {meta["short"]} is best explained by the visible traffic profile plus corridor-specific constraints'
    if variant in ('full', 'flow_cal_wx'):
        txt += ' and, when present, weather pressure'
    if variant in ('full', 'flow_cal_wx', 'flow_cal'):
        txt += ' and calendar pressure'
    if variant == 'full':
        txt += ' with seasonal demand context'
    txt += '.\n[TomTom MOVE dynamic data - monthly values 2023-2024]'
    return txt


def answer_whatif(row, df_ref):
    r = row
    cid = int(r['corr_id'])
    meta = CORR_META[cid]
    mn = MONTH_NAMES_FULL[int(r['month']) - 1]
    yr = int(r['year'])

    rain_cur = float(r['wx_rain_mm_sum'])
    rain_double = rain_cur * 2
    is_monsoon = int(r['is_monsoon'])
    tt_am = float(r['tt_ratio_Weekday_AM1'])
    tt_pm = float(r['tt_ratio_Weekday_PM1'])

    if rain_cur < 50:
        tt_increase = 0.015
        impact_desc = 'minor - extra surface wetness with limited structural disruption'
    elif rain_cur < 150:
        tt_increase = 0.025
        impact_desc = 'moderate - heavier showers would slow traffic noticeably'
    elif rain_cur < 300:
        tt_increase = 0.040
        impact_desc = 'significant - flood-prone sections and queue growth become plausible'
    else:
        tt_increase = 0.055
        impact_desc = 'severe - very high flood risk and strong reliability degradation'

    tt_am_new = tt_am + tt_increase
    tt_pm_new = tt_pm + tt_increase * 1.2

    if cid == 1:
        corridor_specific = 'Patong Hill mountain sections are especially vulnerable to flooding or landslide disruption.'
    elif cid == 2:
        corridor_specific = 'Urban intersections near Chalong are likely to accumulate queues quickly under heavy rain.'
    elif cid == 0:
        corridor_specific = 'Airport Road is flatter, so disruption is more about standing water and visibility than slope failure.'
    else:
        corridor_specific = 'Bypass Road is relatively resilient, but higher rainfall still increases delay through lower speeds and incidents.'

    txt = (
        f'What-if: rainfall doubles on {meta["name"]} in {mn} {yr}\n\n'
        f'Scenario: {rain_cur:.0f} mm -> {rain_double:.0f} mm total monthly precipitation\n\n'
        f'Impact assessment ({impact_desc}):\n\n'
        f'  Current   TT ratio AM={tt_am:.3f}, PM={tt_pm:.3f}\n'
        f'  Estimated TT ratio AM~{tt_am_new:.3f} [{tt_level(tt_am_new)}], PM~{tt_pm_new:.3f} [{tt_level(tt_pm_new)}]\n\n'
        f'Likely traffic effects:\n'
        f'  1. Lower road friction and poorer visibility reduce speeds.\n'
        f'  2. Incident probability rises, creating longer queues and slower recovery.\n'
        f'  3. Reliability worsens first on the most sensitive corridor sections.\n'
        f'  4. {corridor_specific}\n\n'
        f'Context: currently {"in monsoon season" if is_monsoon else "outside monsoon season"}.\n'
        f'Recommendation: if rainfall risk rises sharply, add buffer time and consider {meta["alt"]} when feasible.\n'
        f'[Note: estimates are model-based and represent directional stress on TT ratio.]'
    )
    return txt


def answer_decision(row, df_ref):
    r = row
    cid = int(r['corr_id'])
    meta = CORR_META[cid]
    mn = MONTH_NAMES_FULL[int(r['month']) - 1]
    yr = int(r['year'])

    tt_am  = r['tt_ratio_Weekday_AM1']; sp_am  = r['spd_Weekday_AM1']
    tt_pm  = r['tt_ratio_Weekday_PM1']; sp_pm  = r['spd_Weekday_PM1']
    tt_off = r['tt_ratio_Weekday_Midday']; sp_off = r['spd_Weekday_Midday']
    tt_wk  = r['tt_ratio_Weekend_AM1']; sp_wk  = r['spd_Weekend_AM1']
    pti_m  = r['pti_mean']; pti_mx = r['pti_max']
    season = r['season']

    slots = [
        ('AM Peak (Mon-Fri 07h)', tt_am, sp_am),
        ('Midday   (Mon-Fri 12-13h)', tt_off, sp_off),
        ('Weekend (Sat-Sun 07h)', tt_wk, sp_wk),
        ('PM Peak (Mon-Fri 17h)', tt_pm, sp_pm),
    ]
    slots_sorted = sorted(slots, key=lambda x: x[1])

    best = slots_sorted[0]
    worst = slots_sorted[-1]

    txt = (
        f'Best travel times on {meta["name"]} - {mn} {yr}:\n\n'
        f'Time slot ranking (best to worst traffic conditions; lower TT ratio is better):\n'
    )
    for rank, (label, tt, sp) in enumerate(slots_sorted, 1):
        marker = '[BEST]' if rank == 1 else '[WORST]' if rank == len(slots_sorted) else '      '
        if rank == 1:
            note = 'closest to free-flow - preferred option'
        elif rank == len(slots_sorted):
            note = 'highest delay - avoid if possible'
        else:
            note = 'intermediate delay - acceptable with buffer time'
        txt += f'  {rank}. {marker} {label}\n'
        txt += f'          TT ratio={tt:.3f} [{tt_level(tt)}] | Speed={sp:.1f} km/h | {note}\n'

    txt += (
        f'\nRecommendation:\n'
        f'  Best time  : {best[0]} (TT ratio {best[1]:.3f}, {best[2]:.1f} km/h)\n'
        f'  Avoid      : {worst[0]} (TT ratio {worst[1]:.3f}, {worst[2]:.1f} km/h)\n\n'
        f'Season context: {SEASON_FULL.get(season, season)}\n'
        f'PTI (max): {pti_mx:.2f} - {pti_desc(pti_mx)}\n'
        f'Tip: Even the best time slot on {meta["short"]} has a PTI of {pti_mx:.2f}x. Add buffer time especially during school term and major events.\n'
        f'[TomTom dynamic monthly data - 2023-2024]'
    )
    return txt


def answer_tourism(row, df_ref):
    r = row
    cid = int(r['corr_id'])
    mn = MONTH_NAMES_FULL[int(r['month']) - 1]
    yr = int(r['year'])

    pax = int(r['flt_total_pax'])
    intl = int(r['flt_intl_arrivals'])
    rain = r['wx_rain_mm_sum']
    temp = r['wx_temp_c_mean']
    season = r['season']
    is_monsoon = int(r['is_monsoon'])
    hol = int(r['cal_n_holidays'])
    high_imp = bool(int(r['cal_high_impact']))
    soc = r['soc_phuket']

    all_corr = df_ref[(df_ref['year'] == int(r['year'])) & (df_ref['month'] == int(r['month']))]
    corr_scores = []
    for _, cr in all_corr.iterrows():
        c = int(cr['corr_id'])
        corr_scores.append((cr['tt_ratio_Weekday_AM1'], cr['pti_max'], c, CORR_META[c]['short']))
    corr_scores = sorted(corr_scores, key=lambda x: (x[0], x[1]))
    best_corr = corr_scores[0] if corr_scores else None
    worst_corr = corr_scores[-1] if corr_scores else None
    if best_corr and worst_corr:
        best_tt, best_pti, best_c, best_name = best_corr
        worst_tt, worst_pti, worst_c, worst_name = worst_corr
        route_advice = (
            f'For AM travel this month, prefer {best_name} among the four measured corridors '
            f'(TT ratio {best_tt:.3f}, PTI {best_pti:.2f}). '
            f'Avoid or add buffer on {worst_name} when possible '
            f'(TT ratio {worst_tt:.3f}, PTI {worst_pti:.2f}).'
        )
    else:
        route_advice = 'Compare current TomTom corridor values before choosing a route.'

    if is_monsoon:
        rating = 'fair for traffic (quieter roads, but rain and sea conditions are challenging)'
        advice = 'If visiting in the monsoon, book activities on dry-season-friendly west coast beaches. Traffic is lighter but rain affects road safety on Patong Hill.'
    elif season in ['high_peak', 'high']:
        rating = 'challenging for traffic (peak tourist volumes - roads are busier, plan extra travel time)'
        advice = 'Peak season means all routes are busier. Consider early morning for beach access and avoid the worst measured corridor when possible.'
    else:
        rating = 'good for traffic (shoulder season - moderate volumes, fair weather)'
        advice = 'Shoulder season offers the best balance: reasonable traffic and good weather. Ideal time to visit.'

    txt = (
        f'Traffic perspective for visiting Phuket in {mn} {yr}:\n\n'
        f'Season: {SEASON_FULL.get(season, season)}\n'
        f'Overall traffic rating: {rating}\n\n'
        f'Corridor-by-corridor guide:\n'
    )

    for c in [0, 1, 2, 3]:
        c_row = all_corr[all_corr['corr_id'] == c]
        if len(c_row) > 0:
            cr = c_row.iloc[0]
            tt = cr['tt_ratio_Weekday_AM1']
            pti = cr['pti_max']
            m = CORR_META[c]
            txt += f'  {m["short"]}: {tt_level(tt)} (TT ratio {tt:.3f}, PTI {pti:.2f}) - {m["desc"][:60]}...\n'

    txt += (
        f'\nTourism snapshot - {mn} {yr}:\n'
        f'  HKT passengers: {pax:,} total ({intl:,} international)\n'
        f'  Weather: {rain:.0f} mm rain, {temp:.1f}C avg\n'
        f'  Google Trends "Phuket": {soc:.0f}/100\n'
        f'  Public holidays: {hol} | High-impact event: {"Yes" if high_imp else "No"}\n\n'
        f'Practical advice:\n  {advice}\n'
        f'  {route_advice}\n'
        f'[TomTom dynamic monthly data - 2023-2024]'
    )
    return txt


print('All 6 answer generators defined:')
print('  answer_nowcast  | answer_forecast | answer_explain')
print('  answer_whatif   | answer_decision | answer_tourism')


All 6 answer generators defined:
  answer_nowcast  | answer_forecast | answer_explain
  answer_whatif   | answer_decision | answer_tourism


In [5]:
# -- Prompt builder ------------------------------------------------------------

QUESTION_TEMPLATES = {
    'nowcast' : 'What are the current traffic conditions on {corridor} this month ({month} {year})? Provide a complete status report.',
    'forecast': 'Predict the next-month AM peak TT ratio for {corridor} from {month} {year}. Return JSON only.',
    'explain' : 'Why is traffic at its current level on {corridor} in {month} {year}? Use only the variables visible in the prompt and rank the main contributing factors.',
    'whatif'  : 'How would traffic change on {corridor} in {month} {year} if monthly rainfall doubled from its current level?',
    'decision': 'What is the best time of day to travel on {corridor} in {month} {year}? Compare all time slots and give a recommendation.',
    'tourism' : 'Is {month} {year} a good time to visit Phuket from a traffic perspective? Which routes should tourists use or avoid?',
}

FORECAST_VARIANT_QUESTION_SUFFIX = {
    'full': ' Use the visible traffic state, traffic memory, and light exogenous anchors if shown. Return exactly one JSON object.',
    'forecast_traffic_core': ' Prioritize only the visible traffic state and traffic memory. Return exactly one JSON object.',
    'forecast_lag_focus': ' Prioritize the traffic-memory block over narrative explanation. Return exactly one JSON object.',
    'forecast_profile_only': ' If only the current traffic profile is visible, still return the best numeric forecast as one JSON object.',
    'forecast_minimal_json': ' Return exactly one JSON object and no prose.',
}

ANSWER_FUNCS = {
    'nowcast' : answer_nowcast,
    'forecast': answer_forecast,
    'explain' : answer_explain,
    'whatif'  : answer_whatif,
    'decision': answer_decision,
    'tourism' : answer_tourism,
}

SYSTEM_PROMPT = (
    'You are an expert traffic analyst for Phuket island, Thailand. '
    'You have access only to the information shown in the prompt. '
    'Provide accurate, data-driven answers about traffic conditions, forecasts, and travel recommendations. '
    'Never mention a factor unless it is visible in the prompt, and always cite specific values from the provided data.'
)

BLOCK_RE = re.compile(
    r'(\[(?:SYSTEM|CORRIDOR|TRAFFIC PROFILE|CONTEXT|HISTORY|QUESTION|ANSWER)\])',
    re.MULTILINE
)


def parse_blocks_local(prompt_text):
    parts = BLOCK_RE.split(prompt_text)
    blocks = {}
    current_key = None
    for part in parts:
        if BLOCK_RE.fullmatch(part):
            current_key = part
            blocks[current_key] = ''
        elif current_key is not None:
            blocks[current_key] += part
    return blocks


def reconstruct_prompt_local(blocks):
    return ''.join(f'{k}{v}' for k, v in blocks.items())


def remove_context_lines_local(ctx, keywords):
    kws = [kw.lower() for kw in keywords]
    return '\n'.join(
        line for line in ctx.split('\n')
        if not any(kw in line.lower() for kw in kws)
    )


def strip_context_season_label_local(ctx):
    lines = ctx.split('\n')
    if lines:
        lines[0] = re.sub(r'\s*\([^)]*season[^)]*\)', '', lines[0], flags=re.IGNORECASE)
    return '\n'.join(lines)


def mask_prompt_blocks_local(prompt_text, variant):
    blocks = parse_blocks_local(prompt_text)

    if variant == 'full':
        return prompt_text

    if variant == 'forecast_traffic_core':
        if '[CONTEXT]' in blocks:
            blocks['[CONTEXT]'] = remove_context_lines_local(
                strip_context_season_label_local(blocks['[CONTEXT]']),
                ['tourism', 'passengers', 'pax', 'arrivals', 'hkt',
                 'weather', 'rain', 'temperature', 'wind', 'precipitation', 'mm rain',
                 'trends', 'google', 'search interest', 'holiday', 'event', 'calendar']
            )
        return reconstruct_prompt_local(blocks)

    if variant == 'forecast_lag_focus':
        blocks.pop('[CONTEXT]', None)
        return reconstruct_prompt_local(blocks)

    if variant == 'forecast_profile_only':
        blocks.pop('[CONTEXT]', None)
        blocks.pop('[HISTORY]', None)
        return reconstruct_prompt_local(blocks)

    if variant == 'forecast_minimal_json':
        blocks.pop('[CONTEXT]', None)
        return reconstruct_prompt_local(blocks)

    if variant == 'flow_only':
        blocks.pop('[CONTEXT]', None)
        blocks.pop('[HISTORY]', None)

    elif variant == 'flow_cal':
        if '[CONTEXT]' in blocks:
            blocks['[CONTEXT]'] = remove_context_lines_local(
                strip_context_season_label_local(blocks['[CONTEXT]']),
                ['tourism', 'passengers', 'pax', 'arrivals', 'hkt',
                 'weather', 'rain', 'temperature', 'wind', 'precipitation', 'mm rain', 'km/h wind',
                 'trends', 'google', 'search interest']
            )
        blocks.pop('[HISTORY]', None)

    elif variant == 'flow_cal_wx':
        if '[CONTEXT]' in blocks:
            blocks['[CONTEXT]'] = remove_context_lines_local(
                blocks['[CONTEXT]'],
                ['tourism', 'passengers', 'pax', 'arrivals', 'hkt',
                 'trends', 'google', 'search interest']
            )

    return reconstruct_prompt_local(blocks)


def build_prompt(row, qtype, df_ref, prompt_variant='full'):
    r = row
    cid = int(r['corr_id'])
    meta = CORR_META[cid]
    year = int(r['year'])
    month = int(r['month'])
    mn_full = MONTH_NAMES_FULL[month - 1]

    has_l1 = not pd.isna(r.get('flt_total_pax_l1', float('nan')))
    has_l2 = not pd.isna(r.get('flt_total_pax_l2', float('nan')))
    has_l3 = not pd.isna(r.get('flt_total_pax_l3', float('nan')))

    def fmt_hist(val, digits=3, fallback='N/A'):
        if pd.isna(val):
            return fallback
        return f'{float(val):.{digits}f}'

    def fmt_hist_int(val, fallback='N/A'):
        if pd.isna(val):
            return fallback
        return f'{int(val):,}'

    history_block_default = '  (Insufficient history - first months of dataset; partial or no prior data available)'
    if has_l3:
        m1 = (month - 2) % 12 + 1; y1 = year - (1 if m1 > month else 0)
        history_block_default = (
            f'  Month-1 ({MONTH_NAMES[m1-1]} {y1}): {fmt_hist_int(r["flt_total_pax_l1"])} pax | {fmt_hist(r["wx_rain_mm_sum_l1"], 0)} mm rain | Trends {fmt_hist(r["soc_phuket_l1"], 0)}/100 | Holidays {fmt_hist(r["cal_n_holidays_l1"], 0)}\n'
        )
        if has_l2:
            m2 = (month - 3) % 12 + 1; y2 = year - (1 if m2 > month else 0)
            history_block_default += f'  Month-2 ({MONTH_NAMES[m2-1]} {y2}): {fmt_hist_int(r["flt_total_pax_l2"])} pax | {fmt_hist(r["wx_rain_mm_sum_l2"], 0)} mm rain | Trends {fmt_hist(r["soc_phuket_l2"], 0)}/100\n'
        if has_l3:
            m3 = (month - 4) % 12 + 1; y3 = year - (1 if m3 > month else 0)
            history_block_default += f'  Month-3 ({MONTH_NAMES[m3-1]} {y3}): {fmt_hist_int(r["flt_total_pax_l3"])} pax | {fmt_hist(r["wx_rain_mm_sum_l3"], 0)} mm rain | Trends {fmt_hist(r["soc_phuket_l3"], 0)}/100'

    question = QUESTION_TEMPLATES[qtype].format(corridor=meta['name'], month=mn_full, year=year)
    if qtype == 'forecast':
        question += FORECAST_VARIANT_QUESTION_SUFFIX.get(prompt_variant, '')

    if qtype == 'explain':
        answer = ANSWER_FUNCS[qtype](row, df_ref, variant=prompt_variant if prompt_variant in ('full', 'flow_cal_wx', 'flow_cal', 'flow_only') else 'full')
    elif qtype == 'forecast':
        answer = ANSWER_FUNCS[qtype](row, df_ref, variant='full')
    else:
        answer = ANSWER_FUNCS[qtype](row, df_ref)

    if qtype == 'forecast':
        traffic_profile_block = (
            f'[TRAFFIC PROFILE] (Forecast-critical traffic state; dynamic TomTom MOVE 2023-2024)\n'
            f'  Current AM cluster:\n'
            f'    Weekday_Early TT {r["tt_ratio_Weekday_Early"]:.3f} | AM1 TT {r["tt_ratio_Weekday_AM1"]:.3f} | AM2 TT {r["tt_ratio_Weekday_AM2"]:.3f}\n'
            f'    PrePM TT {r["tt_ratio_Weekday_PrePM"]:.3f} | PM1 TT {r["tt_ratio_Weekday_PM1"]:.3f} | Weekend_AM1 TT {r["tt_ratio_Weekend_AM1"]:.3f}\n'
            f'    AM1 speed {r["spd_Weekday_AM1"]:.1f} km/h | AM1 PTI {r["pti_Weekday_AM1"]:.2f} | PTI max {r["pti_max"]:.2f}\n'
            f'    Shape deltas: AM2-AM1 {r["tt_ratio_Weekday_AM2"] - r["tt_ratio_Weekday_AM1"]:+.3f} | PM1-AM1 {r["tt_ratio_Weekday_PM1"] - r["tt_ratio_Weekday_AM1"]:+.3f} | Weekend-AM1 {r["tt_ratio_Weekend_AM1"] - r["tt_ratio_Weekday_AM1"]:+.3f}'
        )

        if not pd.isna(r.get('tt_ratio_Weekday_AM1_l3', float('nan'))):
            m1 = (month - 2) % 12 + 1; y1 = year - (1 if m1 > month else 0)
            m2 = (month - 3) % 12 + 1; y2 = year - (1 if m2 > month else 0)
            m3 = (month - 4) % 12 + 1; y3 = year - (1 if m3 > month else 0)
            history_block = (
                f'  Month-1 ({MONTH_NAMES[m1-1]} {y1}): AM1 {fmt_hist(r.get("tt_ratio_Weekday_AM1_l1"), 3)} | AM1 speed {fmt_hist(r.get("spd_Weekday_AM1_l1"), 1)} | AM1 PTI {fmt_hist(r.get("pti_Weekday_AM1_l1"), 2)} | AM2 {fmt_hist(r.get("tt_ratio_Weekday_AM2_l1"), 3)} | PrePM {fmt_hist(r.get("tt_ratio_Weekday_PrePM_l1"), 3)} | Weekend_AM1 {fmt_hist(r.get("tt_ratio_Weekend_AM1_l1"), 3)} | pax {fmt_hist_int(r.get("flt_total_pax_l1"))} | rain {fmt_hist(r.get("wx_rain_mm_sum_l1"), 0)} mm\n'
                f'  Month-2 ({MONTH_NAMES[m2-1]} {y2}): AM1 {fmt_hist(r.get("tt_ratio_Weekday_AM1_l2"), 3)} | AM1 speed {fmt_hist(r.get("spd_Weekday_AM1_l2"), 1)} | AM1 PTI {fmt_hist(r.get("pti_Weekday_AM1_l2"), 2)} | AM2 {fmt_hist(r.get("tt_ratio_Weekday_AM2_l2"), 3)} | PrePM {fmt_hist(r.get("tt_ratio_Weekday_PrePM_l2"), 3)} | Weekend_AM1 {fmt_hist(r.get("tt_ratio_Weekend_AM1_l2"), 3)} | pax {fmt_hist_int(r.get("flt_total_pax_l2"))} | rain {fmt_hist(r.get("wx_rain_mm_sum_l2"), 0)} mm\n'
                f'  Month-3 ({MONTH_NAMES[m3-1]} {y3}): AM1 {fmt_hist(r.get("tt_ratio_Weekday_AM1_l3"), 3)} | AM1 speed {fmt_hist(r.get("spd_Weekday_AM1_l3"), 1)} | AM1 PTI {fmt_hist(r.get("pti_Weekday_AM1_l3"), 2)} | AM2 {fmt_hist(r.get("tt_ratio_Weekday_AM2_l3"), 3)} | PrePM {fmt_hist(r.get("tt_ratio_Weekday_PrePM_l3"), 3)} | Weekend_AM1 {fmt_hist(r.get("tt_ratio_Weekend_AM1_l3"), 3)} | pax {fmt_hist_int(r.get("flt_total_pax_l3"))} | rain {fmt_hist(r.get("wx_rain_mm_sum_l3"), 0)} mm'
            )
        else:
            history_block = '  (Traffic-memory lags unavailable in the first months of 2023; use current traffic state and season anchors only.)'

        # ── YoY comparison block ────────────────────────────────────────────
        yoy_row = get_yoy_row(df_ref, cid, year, month)
        if yoy_row is not None:
            yoy_tt  = float(r['tt_ratio_Weekday_AM1']) - float(yoy_row['tt_ratio_Weekday_AM1'])
            yoy_pax = (int(r['flt_total_pax']) - int(yoy_row['flt_total_pax'])) / max(int(yoy_row['flt_total_pax']), 1) * 100
            yoy_rain = float(r['wx_rain_mm_sum']) - float(yoy_row['wx_rain_mm_sum'])
            yoy_line = f'  YoY vs {mn_full} {year-1}: tt_ratio_AM1 {yoy_tt:+.3f} | pax {yoy_pax:+.1f}% | rain {yoy_rain:+.0f} mm'
        else:
            yoy_line = f'  YoY: first year in dataset — no {mn_full} {year-1} baseline available'

        # ── Corridor ranking block ───────────────────────────────────────────
        ranking = get_corridor_ranking(df_ref, year, month)
        rank_pos = next((i+1 for i, x in enumerate(ranking) if x['corr_id'] == cid), None)
        rank_line = (
            f'  Corridor AM1 rank this month: {rank_pos}/4 '
            + '(' + ' | '.join(f'{x["name"]} {x["tt_am1"]:.3f}' for x in ranking) + ')'
        )

        context_block = (
            f'[CONTEXT] {year}-{month:02d} - {mn_full} {year} ({r["season"]} season)\n'
            f'  Calendar / season: {int(r["cal_n_holidays"])} holiday(s) | {int(r["cal_n_events"])} event(s) | High-impact: {"Yes" if int(r["cal_high_impact"]) else "No"}\n'
            f'  Light exogenous anchors: total pax {int(r["flt_total_pax"]):,} | rain {r["wx_rain_mm_sum"]:.0f} mm | wind {r["wx_wind_kmh"]:.1f} km/h\n'
            f'{yoy_line}\n'
            f'{rank_line}'
        )
    else:
        traffic_profile_block = (
            f'[TRAFFIC PROFILE] (Dynamic monthly data - TomTom MOVE 2023-2024; TT ratio 1.0=free-flow, higher=slower)\n'
            f'  AM Peak  (Mon-Fri 07h): TT ratio {r["tt_ratio_Weekday_AM1"]:.3f} | Speed {r["spd_Weekday_AM1"]:.1f} km/h\n'
            f'  PM Peak  (Mon-Fri 17h): TT ratio {r["tt_ratio_Weekday_PM1"]:.3f} | Speed {r["spd_Weekday_PM1"]:.1f} km/h\n'
            f'  Midday   (Mon-Fri 12-13h): TT ratio {r["tt_ratio_Weekday_Midday"]:.3f} | Speed {r["spd_Weekday_Midday"]:.1f} km/h\n'
            f'  Weekend  (Sat-Sun 07h): TT ratio {r["tt_ratio_Weekend_AM1"]:.3f} | Speed {r["spd_Weekend_AM1"]:.1f} km/h\n'
            f'  PTI: {r["pti_mean"]:.2f} (mean), {r["pti_max"]:.2f} (max) - {pti_desc(r["pti_max"])}'
        )
        context_block = (
            f'[CONTEXT] {year}-{month:02d} - {mn_full} {year} ({r["season"]} season)\n'
            f'  Tourism : {int(r["flt_total_pax"]):,} total pax at HKT | {int(r["flt_intl_arrivals"]):,} intl. arrivals\n'
            f'  Weather : {r["wx_rain_mm_sum"]:.0f} mm rain | {r["wx_temp_c_mean"]:.1f}C avg | {r["wx_wind_kmh"]:.1f} km/h wind\n'
            f'  Trends  : Google "Phuket" {r["soc_phuket"]:.0f}/100 | "Phuket flights" {r["soc_phuket_flight"]:.0f}/100\n'
            f'  Calendar: {int(r["cal_n_holidays"])} holiday(s) | {int(r["cal_n_events"])} event(s) | High-impact: {"Yes" if int(r["cal_high_impact"]) else "No"}'
        )
        history_block = history_block_default

    prompt_full = (
        f'[SYSTEM] {SYSTEM_PROMPT}\n\n'
        f'[CORRIDOR] {meta["name"]}\n'
        f'  Length: {int(r["corr_length_km"])} km | Tourist access: {"Yes" if int(r["corr_tourist_access"]) else "No"}\n'
        f'  {meta["desc"]}\n\n'
        f'{traffic_profile_block}\n\n'
        f'{context_block}\n\n'
        f'[HISTORY] Last 3 months:\n{history_block}\n\n'
        f'[QUESTION] {question}\n\n'
        f'[ANSWER] {answer}'
    )

    prompt = mask_prompt_blocks_local(prompt_full, prompt_variant)
    return question, answer, prompt


print('build_prompt() defined.')
print(f'Question templates ({len(QUESTION_TEMPLATES)}):')
for qtype, tmpl in QUESTION_TEMPLATES.items():
    print(f'  [{qtype}] {tmpl[:70]}...')


build_prompt() defined.
Question templates (6):
  [nowcast] What are the current traffic conditions on {corridor} this month ({mon...
  [forecast] Predict the next-month AM peak TT ratio for {corridor} from {month} {y...
  [explain] Why is traffic at its current level on {corridor} in {month} {year}? U...
  [whatif] How would traffic change on {corridor} in {month} {year} if monthly ra...
  [decision] What is the best time of day to travel on {corridor} in {month} {year}...
  [tourism] Is {month} {year} a good time to visit Phuket from a traffic perspecti...


In [6]:
# -- Generate prompts ----------------------------------------------------------
QUESTION_TYPES = list(QUESTION_TEMPLATES.keys())
# V9: forecast-only training, but with several forecast prompt views built from
# current traffic state + traffic-memory lags.
TRAIN_QTYPES_FOR_EXPORT = ['forecast']
FORECAST_TRAIN_VARIANTS = ['forecast_traffic_core', 'forecast_lag_focus', 'forecast_profile_only', 'forecast_minimal_json']
EXPLAIN_TRAIN_VARIANT_POOL = []

def choose_explain_train_variant(row):
    return 'full'

records = []
errors = []

for idx, row in df.iterrows():
    corr_id = int(row['corr_id'])
    year = int(row['year'])
    month = int(row['month'])
    split = row['split']

    for qtype in QUESTION_TYPES:
        include_base = (split != 'train') or (qtype in TRAIN_QTYPES_FOR_EXPORT)
        if include_base:
            try:
                question, answer, prompt = build_prompt(row, qtype, df, prompt_variant='full')
                record = {
                    'id': f'C{corr_id}_{year}_{month:02d}_{qtype}',
                    'corridor_id': row['corridor'],
                    'corr_id': corr_id,
                    'year': year,
                    'month': month,
                    'season': row['season'],
                    'split': split,
                    'question_type': qtype,
                    'prompt_variant': 'full',
                    'question': question,
                    'answer': answer,
                    'prompt': prompt,
                    'has_full_history': not pd.isna(row.get('tt_ratio_Weekday_AM1_l3', float('nan'))),
                    'traffic_is_static': False,
                }
                records.append(record)
            except Exception as e:
                errors.append({'idx': idx, 'qtype': qtype, 'variant': 'full', 'error': str(e)})

        if split == 'train' and qtype == 'forecast':
            for variant in FORECAST_TRAIN_VARIANTS:
                try:
                    question, answer, prompt = build_prompt(row, qtype, df, prompt_variant=variant)
                    record = {
                        'id': f'C{corr_id}_{year}_{month:02d}_{qtype}_{variant}',
                        'corridor_id': row['corridor'],
                        'corr_id': corr_id,
                        'year': year,
                        'month': month,
                        'season': row['season'],
                        'split': split,
                        'question_type': qtype,
                        'prompt_variant': variant,
                        'question': question,
                        'answer': answer,
                        'prompt': prompt,
                        'has_full_history': not pd.isna(row.get('tt_ratio_Weekday_AM1_l3', float('nan'))),
                        'traffic_is_static': False,
                    }
                    records.append(record)
                except Exception as e:
                    errors.append({'idx': idx, 'qtype': qtype, 'variant': variant, 'error': str(e)})

print('=' * 70)
print('PROMPT GENERATION COMPLETE')
print('=' * 70)
print(f'Total prompts generated: {len(records)}')
train_rows = int((df['split'] == 'train').sum())
nontrain_rows = len(df) - train_rows
train_base_total = train_rows * len(TRAIN_QTYPES_FOR_EXPORT)
nontrain_total = nontrain_rows * len(QUESTION_TYPES)
extra_forecast_total = train_rows * len(FORECAST_TRAIN_VARIANTS)
extra_explain_total = train_rows * len(EXPLAIN_TRAIN_VARIANT_POOL)
expected_total = train_base_total + nontrain_total + extra_forecast_total + extra_explain_total
print(f'Expected: nontrain base {nontrain_total} + train base {train_base_total} + forecast augmentation {extra_forecast_total} + explain augmentation {extra_explain_total} = {expected_total}')
print(f'Errors: {len(errors)}')

if errors:
    print('ERRORS:')
    for e in errors[:5]:
        print(f'  {e}')

print()
qtypes_df = pd.DataFrame(records)
print('Distribution by question type:')
print(qtypes_df['question_type'].value_counts().to_string())

print()
print('Distribution by split:')
print(qtypes_df.groupby(['split', 'question_type']).size().unstack(fill_value=0).to_string())

print()
print('Distribution by prompt variant:')
print(qtypes_df['prompt_variant'].value_counts().to_string())

print()
print('Prompt length stats (characters):')
lengths = qtypes_df['prompt'].str.len()
print(f'  mean={lengths.mean():.0f} | std={lengths.std():.0f} | min={lengths.min()} | max={lengths.max()}')
print('Answer length stats (characters):')
a_lengths = qtypes_df['answer'].str.len()
print(f'  mean={a_lengths.mean():.0f} | std={a_lengths.std():.0f} | min={a_lengths.min()} | max={a_lengths.max()}')

print()
print('Prompts with full 3-month history:')
print(f'  {qtypes_df["has_full_history"].sum()} / {len(qtypes_df)} ({qtypes_df["has_full_history"].mean()*100:.1f}%)')
print(f'  Partial history: {(~qtypes_df["has_full_history"]).sum()}')


PROMPT GENERATION COMPLETE
Total prompts generated: 528
Expected: nontrain base 288 + train base 48 + forecast augmentation 192 + explain augmentation 0 = 528
Errors: 0

Distribution by question type:
question_type
forecast    288
nowcast      48
explain      48
whatif       48
decision     48
tourism      48

Distribution by split:
question_type  decision  explain  forecast  nowcast  tourism  whatif
split                                                               
test                 24       24        24       24       24      24
train                 0        0       240        0        0       0
val                  24       24        24       24       24      24

Distribution by prompt variant:
prompt_variant
full                     336
forecast_traffic_core     48
forecast_lag_focus        48
forecast_profile_only     48
forecast_minimal_json     48

Prompt length stats (characters):
  mean=2154 | std=590 | min=1179 | max=2991
Answer length stats (characters):
  mean=522 | s

In [7]:
# ── Print 1 full sample per question type ─────────────────────────────────────
# Use a mid-dataset row (June 2023, Patong Hill) for all 6 types
SAMPLE_YEAR = 2023
SAMPLE_MONTH = 6
SAMPLE_CORR = 1  # Patong Hill — most interesting corridor

print('=' * 70)
print(f'SAMPLE PROMPTS — {MONTH_NAMES_FULL[SAMPLE_MONTH-1]} {SAMPLE_YEAR}, Patong Hill (corridor {SAMPLE_CORR})')
print('=' * 70)

for qtype in QUESTION_TYPES:
    sample_id = f'C{SAMPLE_CORR}_{SAMPLE_YEAR}_{SAMPLE_MONTH:02d}_{qtype}'
    sample_records = [r for r in records if r['id'] == sample_id]
    if not sample_records:
        print(f'\n[{qtype.upper()}] — sample not found (id={sample_id})')
        continue
    rec = sample_records[0]
    print(f'\n{"="*70}')
    print(f'[{qtype.upper()}] ID: {rec["id"]} | Split: {rec["split"]}')
    print(f'{'='*70}')
    print(rec['prompt'])
    print()

SAMPLE PROMPTS — June 2023, Patong Hill (corridor 1)

[NOWCAST] — sample not found (id=C1_2023_06_nowcast)

[FORECAST] ID: C1_2023_06_forecast | Split: train
[SYSTEM] You are an expert traffic analyst for Phuket island, Thailand. You have access only to the information shown in the prompt. Provide accurate, data-driven answers about traffic conditions, forecasts, and travel recommendations. Never mention a factor unless it is visible in the prompt, and always cite specific values from the provided data.

[CORRIDOR] Patong Hill (Route 4029)
  Length: 9 km | Tourist access: Yes
  Mountain pass road from Kathu to Patong Beach — most congested corridor on the island, strong high/low season contrast.

[TRAFFIC PROFILE] (Forecast-critical traffic state; dynamic TomTom MOVE 2023-2024)
  Current AM cluster:
    Weekday_Early TT 0.960 | AM1 TT 1.180 | AM2 TT 1.090
    PrePM TT 1.260 | PM1 TT 1.220 | Weekend_AM1 TT 0.910
    AM1 speed 22.5 km/h | AM1 PTI 2.61 | PTI max 3.33
    Shape deltas: AM2

In [8]:
# ── Detailed statistics ────────────────────────────────────────────────────────
df_prompts = pd.DataFrame(records)

print('=' * 70)
print('DATASET STATISTICS')
print('=' * 70)

print('\n1. Total prompts by corridor x question_type:')
pivot = df_prompts.groupby(['corr_id', 'question_type']).size().unstack(fill_value=0)
pivot.index = [CORR_META[i]['short'] for i in pivot.index]
print(pivot.to_string())

print('\n2. Train/val/test split by question_type:')
print(df_prompts.groupby(['split', 'question_type']).size().unstack(fill_value=0).to_string())

print('\n3. Prompt length by question type (chars):')
df_prompts['prompt_len'] = df_prompts['prompt'].str.len()
df_prompts['answer_len'] = df_prompts['answer'].str.len()
print(df_prompts.groupby('question_type')[['prompt_len', 'answer_len']].describe().round(0).to_string())

print('\n4. Prompts by season:')
print(df_prompts.groupby(['season', 'question_type']).size().unstack(fill_value=0).to_string())

print('\n5. Sample IDs per type (first 3):')
for qtype in QUESTION_TYPES:
    ids = df_prompts[df_prompts['question_type'] == qtype]['id'].head(3).tolist()
    print(f'  {qtype}: {ids}')

DATASET STATISTICS

1. Total prompts by corridor x question_type:
question_type  decision  explain  forecast  nowcast  tourism  whatif
Airport Road         12       12        72       12       12      12
Patong Hill          12       12        72       12       12      12
Town-Rawai           12       12        72       12       12      12
Bypass Road          12       12        72       12       12      12

2. Train/val/test split by question_type:
question_type  decision  explain  forecast  nowcast  tourism  whatif
split                                                               
test                 24       24        24       24       24      24
train                 0        0       240        0        0       0
val                  24       24        24       24       24      24

3. Prompt length by question type (chars):
              prompt_len                                                        answer_len                                                      
            

In [9]:
# ── Export to JSONL ────────────────────────────────────────────────────────────
output_path = '../data/llm_prompts_v10_enriched_yoy_corridors.jsonl'

with open(output_path, 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

size_kb = os.path.getsize(output_path) / 1024

print('=' * 70)
print('EXPORT: llm_prompts.jsonl')
print('=' * 70)
print(f'Path  : {output_path}')
print(f'Lines : {len(records)}')
print(f'Size  : {size_kb:.1f} KB ({size_kb/1024:.2f} MB)')

# Quick spot-check: read back and verify
with open(output_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f'\nRead-back check: {len(lines)} lines')

# Verify each line is valid JSON
parse_errors = 0
for i, line in enumerate(lines):
    try:
        json.loads(line)
    except json.JSONDecodeError as e:
        parse_errors += 1
        print(f'  JSON parse error at line {i}: {e}')

print(f'JSON parse errors: {parse_errors} / {len(lines)}')
if parse_errors == 0:
    print('✓ All lines are valid JSON')

# Sample line
sample = json.loads(lines[100])
print(f'\nSample record (line 100):')
print(f'  id              : {sample["id"]}')
print(f'  corridor_id     : {sample["corridor_id"]}')
print(f'  year-month      : {sample["year"]}-{sample["month"]:02d}')
print(f'  question_type   : {sample["question_type"]}')
print(f'  split           : {sample["split"]}')
print(f'  traffic_is_static: {sample["traffic_is_static"]}')
print(f'  has_full_history: {sample["has_full_history"]}')
print(f'  prompt length   : {len(sample["prompt"])} chars')
print(f'  answer length   : {len(sample["answer"])} chars')

EXPORT: llm_prompts.jsonl
Path  : ../data/llm_prompts_v10_enriched_yoy_corridors.jsonl
Lines : 528
Size  : 1661.1 KB (1.62 MB)

Read-back check: 528 lines
JSON parse errors: 0 / 528
✓ All lines are valid JSON

Sample record (line 100):
  id              : C0_2024_07_decision
  corridor_id     : Airport Road (Route 402)
  year-month      : 2024-07
  question_type   : decision
  split           : test
  traffic_is_static: False
  has_full_history: True
  prompt length   : 2806 chars
  answer length   : 1148 chars


In [10]:
# -- Final validation ----------------------------------------------------------
print('=' * 70)
print('PHASE 2.2 - FINAL VALIDATION')
print('=' * 70)

df_check = pd.DataFrame([json.loads(l) for l in open(output_path)])
print(f'\nJSONL reloaded: {len(df_check)} records')
print(f'Columns: {list(df_check.columns)}')

train_rows = int((df['split'] == 'train').sum())
nontrain_rows = len(df) - train_rows
train_base_total = train_rows * len(TRAIN_QTYPES_FOR_EXPORT)
nontrain_total = nontrain_rows * len(QUESTION_TYPES)
extra_forecast_total = train_rows * len(FORECAST_TRAIN_VARIANTS)
extra_explain_total = train_rows * len(EXPLAIN_TRAIN_VARIANT_POOL)
expected_total = train_base_total + nontrain_total + extra_forecast_total + extra_explain_total
assert len(df_check) == expected_total, f'Expected {expected_total}, got {len(df_check)}'
print(f'\n+ Total count: {len(df_check)} = nontrain base {nontrain_total} + train base {train_base_total} + forecast augmentation {extra_forecast_total} + explain augmentation {extra_explain_total}')

assert df_check['corr_id'].nunique() == 4
print('+ All 4 corridors present')

assert set(df_check['question_type'].unique()) == set(QUESTION_TYPES)
print(f'+ All 6 question types present: {sorted(df_check["question_type"].unique())}')

assert 'prompt_variant' in df_check.columns
print(f'+ prompt_variant field present: {sorted(df_check["prompt_variant"].unique())}')

split_counts = df_check['split'].value_counts()
print(f'\nSplit distribution:')
print(f'  train : {split_counts.get("train", 0):3d} ({split_counts.get("train", 0)/len(df_check)*100:.1f}%)')
print(f'  val   : {split_counts.get("val",   0):3d} ({split_counts.get("val",   0)/len(df_check)*100:.1f}%)')
print(f'  test  : {split_counts.get("test",  0):3d} ({split_counts.get("test",  0)/len(df_check)*100:.1f}%)')

assert (~df_check['traffic_is_static']).all(), 'Some records have traffic_is_static=True!'
print(f'\n+ traffic_is_static=False for all {len(df_check)} records (dynamic TomTom data)')

assert (df_check['prompt'].str.len() > 100).all()
assert (df_check['answer'].str.len() > 50).all()
print(f'\n+ No empty prompts or answers')
print(f'  Prompt: min={df_check["prompt"].str.len().min()} chars, max={df_check["prompt"].str.len().max()} chars')
print(f'  Answer: min={df_check["answer"].str.len().min()} chars, max={df_check["answer"].str.len().max()} chars')

print('\nVariant distribution:')
print(df_check['prompt_variant'].value_counts().to_string())

print()
print('=' * 70)
print('PIPELINE OUTPUT:')
print('  data/llm_prompts.jsonl  ->  input for Phase 4 LoRA fine-tuning')
print('  Fields: id, corridor_id, corr_id, year, month, season, split,')
print('          question_type, prompt_variant, question, answer, prompt,')
print('          has_full_history, traffic_is_static')
print()
print('NOTE: traffic_is_static=False - dynamic TomTom monthly data (2023-2024).')


PHASE 2.2 - FINAL VALIDATION

JSONL reloaded: 528 records
Columns: ['id', 'corridor_id', 'corr_id', 'year', 'month', 'season', 'split', 'question_type', 'prompt_variant', 'question', 'answer', 'prompt', 'has_full_history', 'traffic_is_static']

+ Total count: 528 = nontrain base 288 + train base 48 + forecast augmentation 192 + explain augmentation 0
+ All 4 corridors present
+ All 6 question types present: ['decision', 'explain', 'forecast', 'nowcast', 'tourism', 'whatif']
+ prompt_variant field present: ['forecast_lag_focus', 'forecast_minimal_json', 'forecast_profile_only', 'forecast_traffic_core', 'full']

Split distribution:
  train : 240 (45.5%)
  val   : 144 (27.3%)
  test  : 144 (27.3%)

+ traffic_is_static=False for all 528 records (dynamic TomTom data)

+ No empty prompts or answers
  Prompt: min=1181 chars, max=3001 chars
  Answer: min=61 chars, max=1332 chars

Variant distribution:
prompt_variant
full                     336
forecast_traffic_core     48
forecast_lag_focus  

## Summary

**Output:** `data/llm_prompts.jsonl` - V10 enriched YoY + corridor ranking dataset

| Question type | Train contribution | Val/Test contribution | Total contribution |
|---------------|--------------------|-----------------------|--------------------|
| nowcast       | 48                 | 48                    | 96                 |
| forecast      | 192                | 48                    | 240                |
| explain       | 0                  | 48                    | 48                 |
| whatif        | 0                  | 48                    | 48                 |
| decision      | 0                  | 48                    | 48                 |
| tourism       | 0                  | 48                    | 48                 |

**Expected total:** 528 prompts = 288 full val/test prompts + 96 base train prompts + 144 extra train forecast variants.

**Prompt structure:** `[SYSTEM] + [CORRIDOR] + [TRAFFIC PROFILE] + [CONTEXT] + [HISTORY] + [QUESTION] + [ANSWER]`

**Key flags per record:**
- `traffic_is_static: false` - dynamic TomTom monthly data (2023-2024)
- `has_full_history: true/false` - false for Jan/Feb/Mar 2023 (no L3 lag available)
- `split: train/val/test` - temporal split matches `ml_dataset.csv`
- `prompt_variant: full/forecast_focus/forecast_minimal/forecast_numeric_only` on train forecast rows; `full` elsewhere

**Why this V8 scratch dataset exists:**
- remove V4 adapter carry-over as a possible source of forecast anchoring
- keep enough train supervision to teach domain reading from scratch (`nowcast`)
- concentrate most extra supervision on the numeric forecast target

**Next:** continue to `06_llm_finetuning.ipynb` for a from-scratch forecast-focused LoRA run.
